In [ ]:
!pip install transformers==4.52.4 tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.17.0
    Uninstalling huggingface_hub-1.17.0:
      Successfully uninstalled huggingface_hub-1.17.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/dataset/Rest-Mex_2026_train.csv")


**Data Formatting: Conditional Input Construction**

In [ ]:
def map_sentiment(p):
    p = int(p)
    if p == 1:
        return "muy negativa"
    elif p == 2:
        return "negativa"
    elif p == 3:
        return "neutral"
    elif p == 4:
        return "positiva"
    else:
        return "muy positiva"

def map_type(t):
    t = t.lower()
    if t == "hotel":
        return "hotel"
    elif t == "restaurant":
        return "restaurante"
    else:
        return "atractivo turístico"

def create_input(row):
    sent = map_sentiment(row["Polarity"])
    tipo = map_type(row["Type"])
    return (
        f"Escribe una reseña turística con una opinión {sent} sobre un {tipo} "
        f"ubicado en {row['Town']}, {row['Region']}. "
        f"La reseña debe describir la experiencia del visitante de forma natural y realista."
    )

def create_target(row):
    return row["Review"]

In [ ]:
df["input_text"] = df.apply(create_input, axis=1)
df["target_text"] = df.apply(create_target, axis=1)

In [ ]:
df.head()

,Title,Review,Polarity,Town,Region,Type,input_text,target_text
0,Mi Lugar Favorito!!!!,Excelente lugar para comer y pasar una buena n...,5.0,Sayulita,Nayarit,Restaurant,Escribe una reseña turística con una opinión m...,Excelente lugar para comer y pasar una buena n...
1,lugares interesantes para visitar,"andar mucho, así que un poco difícil para pers...",4.0,Tulum,QuintanaRoo,Attractive,Escribe una reseña turística con una opinión p...,"andar mucho, así que un poco difícil para pers..."
2,No es el mismo Dreams,"Es nuestra cuarta visita a Dreams Tulum, elegi...",3.0,Tulum,QuintanaRoo,Hotel,Escribe una reseña turística con una opinión n...,"Es nuestra cuarta visita a Dreams Tulum, elegi..."
3,un buen panorama cerca de CancÃºn,"Estando en CancÃºn, fuimos al puerto y tomamos...",4.0,Isla_Mujeres,QuintanaRoo,Attractive,Escribe una reseña turística con una opinión p...,"Estando en CancÃºn, fuimos al puerto y tomamos..."
4,El mejor,Es un lugar antiguo y por eso me encanto tiene...,5.0,Patzcuaro,Michoacan,Hotel,Escribe una reseña turística con una opinión m...,Es un lugar antiguo y por eso me encanto tiene...


In [ ]:
len(df)

208051

**Estratificación de Datos**

In [ ]:
from sklearn.model_selection import train_test_split

df_p = df[["input_text", "target_text", "Polarity"]]

#Dividimos el DataFrame usando Scikit-Learn
df_train, df_test = train_test_split( df_p, test_size=0.1, stratify=df["Polarity"], random_state=42)


In [ ]:
len(df_train)

187245

In [ ]:
len(df_test)

20806

In [ ]:
df_test = df_test.rename(columns={'target_text': 'Opinion'})
df_test = df_test[["Opinion", "Polarity"]]
df_test.head()

,Opinion,Polarity
53706,"Fuimos a La Rustica, simplemente buscando una ...",5.0
128757,"Nos encantó, nos fuimos por nuestra cuenta en ...",5.0
99791,"Sierra Norte comida muy típico de la región, e...",4.0
119133,Tenía grandes esperanzas puestas en este hotel...,1.0
35347,Esta vez fui acompañado de una tia y sus amiga...,5.0


In [ ]:
df_test.to_csv('/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/dataset/test_clasificador.csv', index=False)

**Clasificador 3:  Cargar Datos**


---



In [ ]:
import pandas as pd

df_train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/LynxesAI-LKEBUAP-Run3-mt5-base-finetuned-configuration-2-roberta-filtering.csv")
df_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/dataset/test_clasificador.csv")


In [ ]:
df_train = df_train[["Opinion", "Polarity"]]
df_train["Polarity"] = df_train["Polarity"].astype(int)
df_train.head()

,Opinion,Polarity
0,Nos quedamos aquí por la noche. Las bebidas er...,1
1,Este hotel no es un lugar muy bonito. El servi...,1
2,"La comida no es muy buena, la verdad el servic...",1
3,"Cuando fuimos, la comida estaba pésima y el se...",1
4,"El lugar es agradable, pero no tienen que paga...",1


In [ ]:
df_test["Polarity"] = df_test["Polarity"].astype(int)
df_test.head()

,Opinion,Polarity
0,"Fuimos a La Rustica, simplemente buscando una ...",5
1,"Nos encantó, nos fuimos por nuestra cuenta en ...",5
2,"Sierra Norte comida muy típico de la región, e...",4
3,Tenía grandes esperanzas puestas en este hotel...,1
4,Esta vez fui acompañado de una tia y sus amiga...,5


In [ ]:
from datasets import Dataset, DatasetDict

train_ds = Dataset.from_pandas(df_train)
test_ds = Dataset.from_pandas(df_test)

train_ds = train_ds.rename_column("Polarity", "labels")
test_ds = test_ds.rename_column("Polarity", "labels")

dataset = DatasetDict({
    "train": train_ds,
    "test": test_ds
})

dataset

DatasetDict({
    train: Dataset({
        features: ['Opinion', 'labels'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['Opinion', 'labels'],
        num_rows: 20806
    })
})

In [ ]:
def encode_labels(example):
    example["labels"] = example["labels"] - 1
    return example

dataset = dataset.map(encode_labels)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20806 [00:00<?, ? examples/s]

In [ ]:
print(dataset["train"][0])
print(dataset["train"][1])
print(dataset["train"][2])
print(dataset["train"][100])
print(dataset["test"][100])

{'Opinion': 'Nos quedamos aquí por la noche. Las bebidas eran muy fuertes. El servicio era demasiado lento, pero no nos dio un trato. No fue una buena experiencia.', 'labels': 0}
{'Opinion': 'Este hotel no es un lugar muy bonito. El servicio es malo, sin embargo la habitación no estaba limpio, el agua en el baño era fría, no había aire acondicionado. No hay wifi y no hay televisión, pero no tienen wifi, así que solo hay una ventana que te lleva al balcón de la planta baja (una vez que se cambiaba). Cuando llegamos, nos dijeron que queríamos dormir, ya que era un poco molesto para hacer reservación antes de que lleg', 'labels': 0}
{'Opinion': 'La comida no es muy buena, la verdad el servicio malo. Las bebidas están a mal precio. No vale la pena visitar Bacalar', 'labels': 0}
{'Opinion': 'El hotel es bonito, pero cuando llegamos al centro de la ciudad, nos habían cobrado 50 pesos por noche para el desayuno. Aunque nos quedamos en la habitación anteriormente, me hubiera gustado haber pedi

**Tokenización**

In [ ]:
from transformers import AutoTokenizer
model_checkpoint = "vg055/roberta-base-bne-finetuned-TripAdvisorDomainAdaptation"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

In [ ]:
def tokenize_reviews(examples):
    return tokenizer(examples["Opinion"], truncation=True)

In [ ]:
encoded_dataset = dataset.map(tokenize_reviews, batched=True, remove_columns=["Opinion"])
encoded_dataset

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20806 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 20806
    })
})

In [ ]:
encoded_dataset["train"][0]

{'labels': 0,
 'input_ids': [0,
  3085,
  21261,
  1609,
  383,
  332,
  2382,
  68,
  2588,
  12632,
  3370,
  727,
  6959,
  68,
  951,
  1416,
  1233,
  4492,
  15674,
  66,
  623,
  403,
  683,
  3777,
  355,
  8580,
  68,
  1893,
  847,
  405,
  2097,
  2060,
  68,
  2],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

**Training Clasificador**

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = 5
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vg055/roberta-base-bne-finetuned-TripAdvisorDomainAdaptation and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, support = precision_recall_fscore_support(
        labels,
        predictions,
        labels=[0,1,2,3,4],
        zero_division=0
    )

    return {
        "f1_muy_negativo": float(f1[0]),
        "f1_negativo": float(f1[1]),
        "f1_neutro": float(f1[2]),
        "f1_positivo": float(f1[3]),
        "f1_muy_positivo": float(f1[4]),
        "f1_macro": float(np.mean(f1))
    }

In [ ]:
from transformers import (AutoModelForSequenceClassification, Trainer, TrainingArguments)

batch_size = 16
num_train_epochs = 2
logging_steps = len(df_train) // (2 * batch_size * num_train_epochs)


training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/ModeloClasificador_3/checkpoints",

    learning_rate=2e-5,
    weight_decay=0.01,

    num_train_epochs=num_train_epochs,

    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",

    logging_steps=logging_steps,

     report_to="none"

)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Muy Negativo,F1 Negativo,F1 Neutro,F1 Positivo,F1 Muy Positivo,F1 Macro
1,0.337900,0.803477,0.489488,0.010772,0.392997,0.463786,0.829576,0.437324
2,0.193900,0.860704,0.600904,0.223776,0.447946,0.477752,0.841351,0.518346


TrainOutput(global_step=376, training_loss=0.34391272955752433, metrics={'train_runtime': 157.7514, 'train_samples_per_second': 38.035, 'train_steps_per_second': 2.383, 'total_flos': 244181858648544.0, 'train_loss': 0.34391272955752433, 'epoch': 2.0})

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.8607040643692017, 'eval_f1_muy_negativo': 0.6009036144578314, 'eval_f1_negativo': 0.22377622377622378, 'eval_f1_neutro': 0.4479456286685202, 'eval_f1_positivo': 0.477751756440281, 'eval_f1_muy_positivo': 0.8413509983205822, 'eval_f1_macro': 0.5183456443326877, 'eval_runtime': 58.8621, 'eval_samples_per_second': 353.47, 'eval_steps_per_second': 22.103, 'epoch': 2.0}


In [ ]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/ModeloClasificador_3/Modelo")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/ModeloClasificador_3/Modelo")

('/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/ModeloClasificador_3/Modelo/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/ModeloClasificador_3/Modelo/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/ModeloClasificador_3/Modelo/vocab.json',
 '/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/ModeloClasificador_3/Modelo/merges.txt',
 '/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/ModeloClasificador_3/Modelo/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/Rest-Mex2026/ModeloClasificador_3/Modelo/tokenizer.json')